# Chapter 16 Natural Language Processing with RNNs and Attention
## 16.1 Generating Shakespearean Text Using a Character RNN
### 16.1.1 Creating the Training Dataset

In [1]:
import tensorflow as tf
from yarl import URL

url_shakespeare = URL(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
path_file = tf.keras.utils.get_file(
    fname="shakespeare.txt", origin=str(url_shakespeare)
)
with open(path_file) as fp:
    shakespeare_text = fp.read()

print(type(shakespeare_text))
shakespeare_text[:100]

2023-03-01 06:34:00.794997: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


<class 'str'>


'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [2]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts(shakespeare_text)
tokenizer

In [3]:
texts = ["First", "fist"]
tokenizer.texts_to_sequences(texts)

[[20, 6, 9, 8, 3], [20, 6, 8, 3]]

In [4]:
for seq in tokenizer.texts_to_sequences_generator(texts):
    print(seq)

[20, 6, 9, 8, 3]
[20, 6, 8, 3]


In [5]:
tokenizer.texts_to_matrix(texts)

array([[0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.]])

In [6]:
seqs = [[20, 6, 9, 8, 3], [20, 6, 8, 3]]
tokenizer.sequences_to_texts(seqs)

['f i r s t', 'f i s t']

In [7]:
tokenizer.word_counts

OrderedDict([('f', 17567),
             ('i', 57369),
             ('r', 53758),
             ('s', 54219),
             ('t', 74024),
             (' ', 169892),
             ('c', 19443),
             ('z', 554),
             ('e', 100652),
             ('n', 53608),
             (':', 10316),
             ('\n', 40000),
             ('b', 14082),
             ('o', 71279),
             ('w', 21115),
             ('p', 12449),
             ('d', 33447),
             ('a', 63326),
             ('y', 22166),
             ('u', 29897),
             ('h', 54378),
             (',', 19846),
             ('m', 25083),
             ('k', 8672),
             ('.', 7885),
             ('l', 37215),
             ('v', 8591),
             ('?', 2462),
             ("'", 6187),
             ('g', 15755),
             (';', 3628),
             ('!', 2172),
             ('j', 948),
             ('-', 1897),
             ('q', 840),
             ('x', 641),
             ('&', 3),
             ('3',

In [8]:
n_chars = len(tokenizer.word_counts)
n_chars

39

In [9]:
tokenizer.word_docs

defaultdict(int,
            {'f': 17567,
             'i': 57369,
             'r': 53758,
             's': 54219,
             't': 74024,
             ' ': 169892,
             'c': 19443,
             'z': 554,
             'e': 100652,
             'n': 53608,
             ':': 10316,
             '\n': 40000,
             'b': 14082,
             'o': 71279,
             'w': 21115,
             'p': 12449,
             'd': 33447,
             'a': 63326,
             'y': 22166,
             'u': 29897,
             'h': 54378,
             ',': 19846,
             'm': 25083,
             'k': 8672,
             '.': 7885,
             'l': 37215,
             'v': 8591,
             '?': 2462,
             "'": 6187,
             'g': 15755,
             ';': 3628,
             '!': 2172,
             'j': 948,
             '-': 1897,
             'q': 840,
             'x': 641,
             '&': 3,
             '3': 27,
             '$': 1})

In [10]:
tokenizer.word_index

{' ': 1,
 'e': 2,
 't': 3,
 'o': 4,
 'a': 5,
 'i': 6,
 'h': 7,
 's': 8,
 'r': 9,
 'n': 10,
 '\n': 11,
 'l': 12,
 'd': 13,
 'u': 14,
 'm': 15,
 'y': 16,
 'w': 17,
 ',': 18,
 'c': 19,
 'f': 20,
 'g': 21,
 'b': 22,
 'p': 23,
 ':': 24,
 'k': 25,
 'v': 26,
 '.': 27,
 "'": 28,
 ';': 29,
 '?': 30,
 '!': 31,
 '-': 32,
 'j': 33,
 'q': 34,
 'x': 35,
 'z': 36,
 '3': 37,
 '&': 38,
 '$': 39}

In [11]:
tokenizer.document_count

1115394

In [12]:
import numpy as np

[text_encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1
text_encoded

array([19,  5,  8, ..., 20, 26, 10])

### 16.1.2 How to Split a Sequential Dataset

In [13]:
size_dataset = tokenizer.document_count
size_dataset

1115394

In [14]:
size_train = round(size_dataset * 0.9)
size_valid = round(size_dataset * 0.1)
size_test = size_dataset - size_train - size_valid
size_train

1003855

In [15]:
dataset_train = tf.data.Dataset.from_tensor_slices(text_encoded[:size_train])
dataset_valid = tf.data.Dataset.from_tensor_slices(text_encoded[size_train:-size_test])
dataset_test = tf.data.Dataset.from_tensor_slices(text_encoded[-size_test:])

for _ in range(2):
    for (
        i_instance,
        instance,
    ) in enumerate(dataset_train.take(5)):
        print(i_instance, instance)

0 tf.Tensor(19, shape=(), dtype=int64)
1 tf.Tensor(5, shape=(), dtype=int64)
2 tf.Tensor(8, shape=(), dtype=int64)
3 tf.Tensor(7, shape=(), dtype=int64)
4 tf.Tensor(2, shape=(), dtype=int64)
0 tf.Tensor(19, shape=(), dtype=int64)
1 tf.Tensor(5, shape=(), dtype=int64)
2 tf.Tensor(8, shape=(), dtype=int64)
3 tf.Tensor(7, shape=(), dtype=int64)
4 tf.Tensor(2, shape=(), dtype=int64)


2023-03-01 06:34:57.373083: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-01 06:34:57.474117: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-01 06:34:57.474453: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-03-01 06:34:57.476036: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operation

### 16.1.3 Chopping the Sequential Dataset into Multiple Windows

##### Test flat_map and interleave

In [16]:
a = [[1, 2, 3, 4], [10, 11, 12, 13], [20, 21, 22, 23]]
ds = tf.data.Dataset.from_tensor_slices(a)
print(list(ds.as_numpy_iterator()))
list(ds.flat_map(lambda x: tf.data.Dataset.from_tensor_slices(x)).as_numpy_iterator())

[array([1, 2, 3, 4], dtype=int32), array([10, 11, 12, 13], dtype=int32), array([20, 21, 22, 23], dtype=int32)]
Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


[1, 2, 3, 4, 10, 11, 12, 13, 20, 21, 22, 23]

In [17]:
list(
    ds.flat_map(
        lambda x: tf.data.Dataset.from_tensor_slices(x).batch(4)
    ).as_numpy_iterator()
)

[array([1, 2, 3, 4], dtype=int32),
 array([10, 11, 12, 13], dtype=int32),
 array([20, 21, 22, 23], dtype=int32)]

In [18]:
list(
    ds.interleave(tf.data.Dataset.from_tensor_slices, cycle_length=3, block_length=None)
)

[<tf.Tensor: shape=(), dtype=int32, numpy=1>,
 <tf.Tensor: shape=(), dtype=int32, numpy=10>,
 <tf.Tensor: shape=(), dtype=int32, numpy=20>,
 <tf.Tensor: shape=(), dtype=int32, numpy=2>,
 <tf.Tensor: shape=(), dtype=int32, numpy=11>,
 <tf.Tensor: shape=(), dtype=int32, numpy=21>,
 <tf.Tensor: shape=(), dtype=int32, numpy=3>,
 <tf.Tensor: shape=(), dtype=int32, numpy=12>,
 <tf.Tensor: shape=(), dtype=int32, numpy=22>,
 <tf.Tensor: shape=(), dtype=int32, numpy=4>,
 <tf.Tensor: shape=(), dtype=int32, numpy=13>,
 <tf.Tensor: shape=(), dtype=int32, numpy=23>]

##### Back to the Shakespeare

In [19]:
n_steps = 100
shift = 1
window_length = n_steps + shift
dataset_train_window = dataset_train.window(
    size=window_length, shift=shift, drop_remainder=True
)
dataset_valid_window = dataset_valid.window(
    size=window_length, shift=shift, drop_remainder=True
)
dataset_test_window = dataset_test.window(
    size=window_length, shift=shift, drop_remainder=True
)

for window in dataset_train_window.take(3):
    print(window)

<_VariantDataset element_spec=TensorSpec(shape=(), dtype=tf.int64, name=None)>
<_VariantDataset element_spec=TensorSpec(shape=(), dtype=tf.int64, name=None)>
<_VariantDataset element_spec=TensorSpec(shape=(), dtype=tf.int64, name=None)>


In [20]:
np.array(list(window.as_numpy_iterator()))

2023-03-01 06:35:30.797709: W tensorflow/core/framework/dataset.cc:769] Input of Window will not be optimized because the dataset does not implement the AsGraphDefInternal() method needed to apply optimizations.


array([ 8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 21,  1, 19,  3,
        8,  1,  0, 16,  1,  0, 22,  8,  3, 18,  1,  1, 12,  0,  4,  9, 15,
        0, 19, 13,  8,  2,  6,  1,  8, 17,  0,  6,  1,  4,  8,  0, 14,  1,
        0,  7, 22,  1,  4, 24, 26, 10, 10,  4, 11, 11, 23, 10,  7, 22,  1,
        4, 24, 17,  0,  7, 22,  1,  4, 24, 26, 10, 10, 19,  5,  8,  7,  2,
        0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 15,  3, 13,  0,  4,  8])

In [21]:
dataset_train_flatten = dataset_train_window.flat_map(
    lambda window: window.batch(window_length)
)
dataset_valid_flatten = dataset_valid_window.flat_map(
    lambda window: window.batch(window_length)
)
dataset_test_flatten = dataset_test_window.flat_map(
    lambda window: window.batch(window_length)
)
for window in dataset_train_flatten.take(3):
    print(window)

tf.Tensor(
[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
  0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
  4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24
 17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23
 10 15  3 13  0], shape=(101,), dtype=int64)
tf.Tensor(
[ 5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1  0
 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1  4
  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24 17
  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10
 15  3 13  0  4], shape=(101,), dtype=int64)
tf.Tensor(
[ 8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1  0 22
  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1  4  8
  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24 17  0
  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 15
 

In [22]:
batch_size = 32
dataset_train_batch = (
    dataset_train_flatten.shuffle(10000)
    .batch(batch_size)
    .map(lambda window: (window[:, :-1], window[:, 1:]))
)
dataset_valid_batch = (
    dataset_valid_flatten.shuffle(10000)
    .batch(batch_size)
    .map(lambda window: (window[:, :-1], window[:, 1:]))
)
dataset_test_batch = (
    dataset_test_flatten.shuffle(10000)
    .batch(batch_size)
    .map(lambda window: (window[:, :-1], window[:, 1:]))
)
for batch in dataset_train_batch.take(3):
    print(batch)

(<tf.Tensor: shape=(32, 100), dtype=int64, numpy=
array([[10,  7,  1, ...,  1,  0, 19],
       [ 7,  1, 18, ...,  1, 27,  7],
       [22,  8,  3, ...,  1,  8,  0],
       ...,
       [24,  1,  0, ..., 16,  4, 15],
       [12,  0, 11, ..., 18,  3, 13],
       [ 2,  0, 18, ...,  4,  8,  1]])>, <tf.Tensor: shape=(32, 100), dtype=int64, numpy=
array([[ 7,  1, 18, ...,  0, 19,  4],
       [ 1, 18,  3, ..., 27,  7,  0],
       [ 8,  3, 18, ...,  8,  0,  2],
       ...,
       [ 1,  0,  4, ...,  4, 15,  0],
       [ 0, 11,  1, ...,  3, 13, 11],
       [ 0, 18,  5, ...,  8,  1,  0]])>)
(<tf.Tensor: shape=(32, 100), dtype=int64, numpy=
array([[ 2,  0,  4, ...,  0, 21,  1],
       [ 1,  0, 19, ...,  6,  3, 16],
       [12,  0,  4, ..., 11,  3,  3],
       ...,
       [ 9,  3,  2, ..., 16,  1,  8],
       [ 0,  7,  1, ...,  1,  2,  5],
       [ 0,  4,  2, ...,  1, 15, 27]])>, <tf.Tensor: shape=(32, 100), dtype=int64, numpy=
array([[ 0,  4, 10, ..., 21,  1,  0],
       [ 0, 19, 11, ...,  3, 16,  0

In [23]:
tf.one_hot([0, 1, 2, 3], depth=3)

<tf.Tensor: shape=(4, 3), dtype=float32, numpy=
array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 0.]], dtype=float32)>

In [24]:
tf.one_hot([[0, 1, 2, 3]] * 3, depth=4)

<tf.Tensor: shape=(3, 4, 4), dtype=float32, numpy=
array([[[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]],

       [[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]],

       [[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]]], dtype=float32)>

In [25]:
dataset_train_one_hot = dataset_train_batch.map(
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=n_chars), Y_batch)
)
dataset_valid_one_hot = dataset_valid_batch.map(
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=n_chars), Y_batch)
)
dataset_test_one_hot = dataset_test_batch.map(
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=n_chars), Y_batch)
)
for batch in dataset_train_one_hot.take(2):
    print(batch)

(<tf.Tensor: shape=(32, 100, 39), dtype=float32, numpy=
array([[[0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]],

       [[0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        ...,
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0.

In [26]:
dataset_train_prefetch = dataset_train_one_hot.prefetch(1)
dataset_valid_prefetch = dataset_valid_one_hot.prefetch(1)
dataset_test_prefetch = dataset_test_one_hot.prefetch(1)
for batch in dataset_train_prefetch.take(1):
    print(batch)

(<tf.Tensor: shape=(32, 100, 39), dtype=float32, numpy=
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1.

In [27]:
ds = tf.data.Dataset.range(12).shuffle(10)
print(list(ds.take(3).as_numpy_iterator()))
print(list(ds.prefetch(1).take(3).as_numpy_iterator()))

[4, 3, 6]
[2, 9, 8]


### 16.1.4 Building and Training the Char-RNN Model

In [27]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.GRU(
            128, dropout=0.2, return_sequences=True, input_shape=(None, n_chars)
        ),
        tf.keras.layers.GRU(128, dropout=0.2, return_sequences=True),
        tf.keras.layers.Dense(n_chars, activation="softmax"),
    ]
)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, None, 128)         64896     
                                                                 
 gru_1 (GRU)                 (None, None, 128)         99072     
                                                                 
 dense (Dense)               (None, None, 39)          5031      
                                                                 
Total params: 168,999
Trainable params: 168,999
Non-trainable params: 0
_________________________________________________________________


In [28]:
import time
from pathlib import Path

root_logdir = Path().absolute() / "logs"

%load_ext tensorboard
%tensorboard --logdir=./logs --port=6006

Launching TensorBoard...

In [29]:
epochs = 20
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=10, restore_best_weights=True
)
tensorboard_cb = tf.keras.callbacks.TensorBoard(
    root_logdir / time.strftime("run_%Y_%m_%d-%H_%M_%S")
)

# history = model.fit(dataset_train_prefetch, validation_data=dataset_valid_prefetch, epochs=epochs, validation_steps=100, callbacks=[early_stopping_cb])
history = model.fit(dataset_train_prefetch, epochs=10)

Epoch 1/10


2023-02-24 06:52:28.069851: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8401


31368/31368 [==============================] - 1190s 38ms/step - loss: 1.6185
Epoch 2/10
31368/31368 [==============================] - 1390s 44ms/step - loss: 1.5395
Epoch 3/10
31368/31368 [==============================] - 1336s 43ms/step - loss: 1.5192
Epoch 4/10
31368/31368 [==============================] - 1038s 33ms/step - loss: 1.5079
Epoch 5/10
31368/31368 [==============================] - 925s 29ms/step - loss: 1.5002
Epoch 6/10
31368/31368 [==============================] - 885s 28ms/step - loss: 1.4952
Epoch 7/10
31368/31368 [==============================] - 866s 28ms/step - loss: 1.4913
Epoch 8/10
31368/31368 [==============================] - 866s 28ms/step - loss: 1.4882
Epoch 9/10
31368/31368 [==============================] - 858s 27ms/step - loss: 1.4857
Epoch 10/10
31368/31368 [==============================] - 851s 27ms/step - loss: 1.4835


In [30]:
dir_models = Path() / "models"
model.save(dir_models / "model")

INFO:tensorflow:Assets written to: models/model/assets


INFO:tensorflow:Assets written to: models/model/assets


### 16.1.5 Using the Char-RNN Model

In [28]:
import numpy as np
from typing import Sequence


def preprocess(texts: Sequence[str]) -> tf.Tensor:
    return tf.one_hot(np.array(tokenizer.texts_to_sequences(texts)) - 1, depth=n_chars)


text = "How are yo"
text_prep = preprocess([text])
text_prep

<tf.Tensor: shape=(1, 10, 39), dtype=float32, numpy=
array([[[0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.

In [29]:
import tensorflow as tf
from pathlib import Path

path_model = Path() / "models" / "model"
model = tf.keras.models.load_model(path_model)
model

In [33]:
y_pred = model.predict(text_prep)
y_pred.shape

1/1 [==============================] - 2s 2s/step


2023-02-25 09:06:14.985486: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8401


(1, 10, 39)

In [34]:
y_class = np.argmax(y_pred, axis=1)
y_class

array([[6, 5, 7, 8, 0, 3, 3, 4, 1, 4, 2, 1, 3, 9, 1, 7, 1, 2, 3, 7, 7, 7,
        1, 2, 7, 1, 2, 2, 2, 2, 2, 2, 7, 7, 3, 1, 3, 0, 1]])

In [35]:
tokenizer.sequences_to_texts(y_class)[0][-1]

' '

In [36]:
np.array(tokenizer.texts_to_sequences(texts))

/tmp/ipykernel_9715/3769257489.py:1: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  np.array(tokenizer.texts_to_sequences(texts))


array([list([20, 6, 9, 8, 3]), list([20, 6, 8, 3])], dtype=object)

In [21]:
def predict_next(text: str, temperature: float = 1.0) -> str:
    y_proba = model(preprocess([text]))[0, -1:]
    logits = tf.math.log(y_proba) / temperature
    id_char = tf.random.categorical(logits, num_samples=1) + 1
    return tokenizer.sequences_to_texts(id_char.numpy())[0]

In [101]:
predict_next("Will you lo")

'v'

In [102]:
predict_next("love and pea")

'r'

In [103]:
predict_next("shall we tal")

'k'

##### `tf.log`

In [45]:
tf.math.log(tf.math.exp(1.0))

<tf.Tensor: shape=(), dtype=float32, numpy=1.0>

In [50]:
tf.random.categorical([[0.1, 0.2, 0.9]], num_samples=10)

<tf.Tensor: shape=(1, 10), dtype=int64, numpy=array([[2, 0, 1, 0, 1, 2, 2, 2, 2, 2]])>

In [51]:
s = "How are yo"
s += "u"
s

'How are you'

##### Back to Shakespeare

In [25]:
from functools import reduce


def predict_text(text: str, n_chars: int = 50, temperature: int = 1) -> str:
    for _ in range(n_chars):
        text += predict_next(text)
    return text


text = predict_text("Do yo")
print(text)

2023-02-28 06:03:06.951352: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8401


Do you, screcl.
i'll prove the teaks to have i being to
Do you mear? have i not gentleman to her ance it of tha


### 16.1.6 Stateful RNN

In [30]:
dataset_window = dataset_train.window(
    size=window_length, shift=n_steps, drop_remainder=True
)
for instance in dataset_window.take(5):
    print(np.array(list(instance.as_numpy_iterator())))

[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
  0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
  4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24
 17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23
 10 15  3 13  0]
[ 0  4  8  1  0  4 11 11  0  8  1  7  3 11 25  1 12  0  8  4  2  6  1  8
  0  2  3  0 12  5  1  0  2  6  4  9  0  2  3  0 19  4 14  5  7  6 29 10
 10  4 11 11 23 10  8  1  7  3 11 25  1 12 26  0  8  1  7  3 11 25  1 12
 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 19  5  8  7  2 17
  0 15  3 13  0]
[ 0 24  9  3 16  0 18  4  5 13  7  0 14  4  8 18  5 13  7  0  5  7  0 18
  6  5  1 19  0  1  9  1 14 15  0  2  3  0  2  6  1  0 22  1  3 22 11  1
 26 10 10  4 11 11 23 10 16  1  0 24  9  3 16 27  2 17  0 16  1  0 24  9
  3 16 27  2 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 11  1
  2  0 13  7  0]
[ 0 24  5 11 11  0  6  5 14 17  0  4  9 12  0 16  1 27 11 11  0  6  4 25


2023-03-01 06:39:41.050939: W tensorflow/core/framework/dataset.cc:769] Input of Window will not be optimized because the dataset does not implement the AsGraphDefInternal() method needed to apply optimizations.


In [31]:
dataset_flat = dataset_window.flat_map(lambda window: window.batch(window_length))
for instance in dataset_flat.take(5):
    print(instance)

tf.Tensor(
[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
  0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
  4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24
 17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23
 10 15  3 13  0], shape=(101,), dtype=int64)
tf.Tensor(
[ 0  4  8  1  0  4 11 11  0  8  1  7  3 11 25  1 12  0  8  4  2  6  1  8
  0  2  3  0 12  5  1  0  2  6  4  9  0  2  3  0 19  4 14  5  7  6 29 10
 10  4 11 11 23 10  8  1  7  3 11 25  1 12 26  0  8  1  7  3 11 25  1 12
 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 19  5  8  7  2 17
  0 15  3 13  0], shape=(101,), dtype=int64)
tf.Tensor(
[ 0 24  9  3 16  0 18  4  5 13  7  0 14  4  8 18  5 13  7  0  5  7  0 18
  6  5  1 19  0  1  9  1 14 15  0  2  3  0  2  6  1  0 22  1  3 22 11  1
 26 10 10  4 11 11 23 10 16  1  0 24  9  3 16 27  2 17  0 16  1  0 24  9
  3 16 27  2 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 11  1
 

In [32]:
dataset_batch = dataset_flat.batch(1)
for instance in dataset_batch.take(3):
    print(instance)

tf.Tensor(
[[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
   0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
   4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24
  17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23
  10 15  3 13  0]], shape=(1, 101), dtype=int64)
tf.Tensor(
[[ 0  4  8  1  0  4 11 11  0  8  1  7  3 11 25  1 12  0  8  4  2  6  1  8
   0  2  3  0 12  5  1  0  2  6  4  9  0  2  3  0 19  4 14  5  7  6 29 10
  10  4 11 11 23 10  8  1  7  3 11 25  1 12 26  0  8  1  7  3 11 25  1 12
  26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 19  5  8  7  2 17
   0 15  3 13  0]], shape=(1, 101), dtype=int64)
tf.Tensor(
[[ 0 24  9  3 16  0 18  4  5 13  7  0 14  4  8 18  5 13  7  0  5  7  0 18
   6  5  1 19  0  1  9  1 14 15  0  2  3  0  2  6  1  0 22  1  3 22 11  1
  26 10 10  4 11 11 23 10 16  1  0 24  9  3 16 27  2 17  0 16  1  0 24  9
   3 16 27  2 26 10 10 19  5  8  7  2  0 18  5  2  5 35

In [33]:
dataset_split = dataset_batch.map(lambda windows: (windows[:, :-1], windows[:, 1:]))
for instance in dataset_split.take(3):
    print(instance)

(<tf.Tensor: shape=(1, 100), dtype=int64, numpy=
array([[19,  5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 21,
         1, 19,  3,  8,  1,  0, 16,  1,  0, 22,  8,  3, 18,  1,  1, 12,
         0,  4,  9, 15,  0, 19, 13,  8,  2,  6,  1,  8, 17,  0,  6,  1,
         4,  8,  0, 14,  1,  0,  7, 22,  1,  4, 24, 26, 10, 10,  4, 11,
        11, 23, 10,  7, 22,  1,  4, 24, 17,  0,  7, 22,  1,  4, 24, 26,
        10, 10, 19,  5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23,
        10, 15,  3, 13]])>, <tf.Tensor: shape=(1, 100), dtype=int64, numpy=
array([[ 5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 21,  1,
        19,  3,  8,  1,  0, 16,  1,  0, 22,  8,  3, 18,  1,  1, 12,  0,
         4,  9, 15,  0, 19, 13,  8,  2,  6,  1,  8, 17,  0,  6,  1,  4,
         8,  0, 14,  1,  0,  7, 22,  1,  4, 24, 26, 10, 10,  4, 11, 11,
        23, 10,  7, 22,  1,  4, 24, 17,  0,  7, 22,  1,  4, 24, 26, 10,
        10, 19,  5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10,
        15,

In [34]:
dataset_onehot = dataset_split.map(
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=n_chars), Y_batch)
)
for instance in dataset_onehot.take(1):
    print(instance)

(<tf.Tensor: shape=(1, 100, 39), dtype=float32, numpy=
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], dtype=float32)>, <tf.Tensor: shape=(1, 100), dtype=int64, numpy=
array([[ 5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10, 21,  1,
        19,  3,  8,  1,  0, 16,  1,  0, 22,  8,  3, 18,  1,  1, 12,  0,
         4,  9, 15,  0, 19, 13,  8,  2,  6,  1,  8, 17,  0,  6,  1,  4,
         8,  0, 14,  1,  0,  7, 22,  1,  4, 24, 26, 10, 10,  4, 11, 11,
        23, 10,  7, 22,  1,  4, 24, 17,  0,  7, 22,  1,  4, 24, 26, 10,
        10, 19,  5,  8,  7,  2,  0, 18,  5,  2,  5, 35,  1,  9, 23, 10,
        15,  3, 13,  0]])>)


In [36]:
dataset_prefetch = dataset_onehot.prefetch(1)
# for instance in dataset_onehot.take(1):
#     print(instance)

In [40]:
for instance in dataset_batch.map(lambda *windows: tf.stack(windows)).take(3):
    print(instance)

tf.Tensor(
[[[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16
    1  0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0
    6  1  4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22
    1  4 24 17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5
   35  1  9 23 10 15  3 13  0]]], shape=(1, 1, 101), dtype=int64)
tf.Tensor(
[[[ 0  4  8  1  0  4 11 11  0  8  1  7  3 11 25  1 12  0  8  4  2  6  1
    8  0  2  3  0 12  5  1  0  2  6  4  9  0  2  3  0 19  4 14  5  7  6
   29 10 10  4 11 11 23 10  8  1  7  3 11 25  1 12 26  0  8  1  7  3 11
   25  1 12 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 19  5
    8  7  2 17  0 15  3 13  0]]], shape=(1, 1, 101), dtype=int64)
tf.Tensor(
[[[ 0 24  9  3 16  0 18  4  5 13  7  0 14  4  8 18  5 13  7  0  5  7  0
   18  6  5  1 19  0  1  9  1 14 15  0  2  3  0  2  6  1  0 22  1  3 22
   11  1 26 10 10  4 11 11 23 10 16  1  0 24  9  3 16 27  2 17  0 16  1
    0 24  9  3 16 27  2 26 10 10 19  5  8  